# Parasites: exploración del dataset

Ejecuta primero los scripts de descarga y preparación del README. Las imágenes y anotaciones son comunes a Faster R-CNN v2 y YOLOv10-S.


In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from matplotlib.patches import Rectangle

from parasite_detection.paths import DATA

if not (DATA / "summary.json").exists():
    raise FileNotFoundError("Con el entorno activado, ejecuta python scripts/prepare_dataset.py desde la raíz del repo.")

summary = json.loads((DATA / "summary.json").read_text())
pd.DataFrame({split: {"images": info["images"], "objects": info["objects"]}
              for split, info in summary.items()}).T


## Distribución de especies

Las clases se cuentan por objetos anotados, no por imágenes.


In [ ]:
counts = pd.DataFrame({split: info["objects_per_class"] for split, info in summary.items()}).fillna(0).astype(int)
display(counts)
counts.plot.barh(figsize=(10, 5), title="Objetos por especie y partición")
plt.xlabel("Objetos")
plt.tight_layout()


## Ejemplos de entrenamiento

Se muestran cajas COCO originales. Cambia `seed` para explorar otros ejemplos sin modificar las particiones.


In [ ]:
import random

split, seed = "train", 42
coco = json.loads((DATA / "annotations" / f"instances_{split}.json").read_text())
classes = {c["id"]: c["name"] for c in coco["categories"]}
annotations = {}
for ann in coco["annotations"]:
    annotations.setdefault(ann["image_id"], []).append(ann)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, item in zip(axes.flat, random.Random(seed).sample(coco["images"], 6)):
    with Image.open(DATA / "images" / split / item["file_name"]) as image:
        ax.imshow(image)
    for ann in annotations.get(item["id"], []):
        x, y, width, height = ann["bbox"]
        ax.add_patch(Rectangle((x, y), width, height, fill=False, edgecolor="lime", linewidth=2))
        ax.text(x, y, classes[ann["category_id"]], color="black", fontsize=8,
                bbox={"facecolor": "lime", "alpha": 0.8, "pad": 1})
    ax.set_title(item["file_name"])
    ax.axis("off")
plt.tight_layout()


## Correspondencia de etiquetas

COCO y torchvision: 1–8 (0 reservado a background). YOLO: 0–7. Ambos formatos contienen los mismos objetos.


In [ ]:
pd.DataFrame(json.loads((DATA / "class_mapping.json").read_text()))
